In [1]:
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import box

# ==============================================================================
# 1. BASE BUILDING PARAMETERS (Modify any value here to update all 30 buildings)
# ==============================================================================
BASE_MFH_CONFIG = {
    # Footprint Dimensions & Geometry (20m x 15m = 300 m²)
    "width_m": 20.0,
    "depth_m": 15.0,
    "grundflaeche_uom": "m2",
    
    # Storeys & Roof
    "anzahlDerOberirdischenGeschosse": 4.0,
    "anzahlDerUnterirdischenGeschosse": 1.0,
    "dachform": 1000.0,                 # 1000: Flat roof, 2100: Saddle/Gable
    "dachart": "Flachdach",
    "bauweise": 2000.0,                 # 2000: Massivbau / geschlossene Bauweise
    "hochhaus": 0.0,
    
    # German ALKIS / DistrictGenerator Typology Classifications
    "gebaeudefunktion": 1020,           # ALKIS code for Mehrfamilienhaus
    "building": "MFH",
    "building_type": "MFH",
    "art": "MFH",
    "advStandardModell": "DLKM",
    "beginnt": "2026-01-01T00:00:00",
    
    # Solar PV Area (Keep well below footprint so it never exceeds roof area in DG)
    "solar_pvarea_value": 45.0,         # 45 m² PV array
    "solar_power_value": 9.0,           # ~9.0 kWp (~0.2 kWp/m²)
    "solar_pvarea_zone_total": 1350.0,
    "solar_power_zone_total": 270.0,
    "solar_n_buildings_in_zone": 30.0,
    "solar_match_score": 1.0,
    "eignung": "Sehr gut geeignet",
    
    # Standard District Consumption Estimates
    "elec_value": 38400.0,              # kWh/a (approx 32 kWh/m²a gross area)
    "elec_zone_total": 1152000.0,
    "elec_n_buildings_in_zone": 30.0,
    "elec_match_score": 1.0,
    "heat_value": 96000.0,              # kWh/a (approx 80 kWh/m²a gross area)
    "heat_zone_total": 2880000.0,
    "heat_n_buildings_in_zone": 30.0,
    "heat_match_score": 1.0,
    
    # Confidence Flags
    "solar_high_confidence": True,
    "elec_high_confidence": True,
    "heat_high_confidence": True,
}

# ==============================================================================
# 2. TABULA GERMAN MFH CLASSES (A to J) & RETROFIT LEVELS
# ==============================================================================
TABULA_MFH_CLASSES = {
    "A": {"era": "<= 1859",     "year": 1850},
    "B": {"era": "1860 - 1918", "year": 1900},
    "C": {"era": "1919 - 1948", "year": 1930},
    "D": {"era": "1949 - 1957", "year": 1954},
    "E": {"era": "1958 - 1968", "year": 1963},
    "F": {"era": "1969 - 1978", "year": 1973},
    "G": {"era": "1979 - 1983", "year": 1981},
    "H": {"era": "1984 - 1994", "year": 1989},
    "I": {"era": "1995 - 2001", "year": 1998},
    "J": {"era": "2002 - 2009", "year": 2005},
}

RETROFIT_LEVELS = [0, 1, 2]

# Spatial grid settings (10 columns along X, 3 rows along Y)
GRID_ORIGIN = (565000.0, 5935000.0)     # EPSG:25832 anchor (Hamburg UTM 32N)
GRID_SPACING_X = 45.0                   # Distance between building centers in meters
GRID_SPACING_Y = 40.0
CRS = "EPSG:25832"

In [2]:
def generate_30_mfh_dataset(
    base_params: dict,
    classes: dict,
    retrofit_levels: list[int],
    origin: tuple[float, float],
    spacing_x: float,
    spacing_y: float,
    crs: str
) -> gpd.GeoDataFrame:
    """Creates exactly 30 MFH buildings with identical parameters except year and retrofit."""
    records = []
    width = base_params["width_m"]
    depth = base_params["depth_m"]
    ground_area = round(width * depth, 2)
    
    col_keys = list(classes.keys())
    
    for col_idx, tabula_key in enumerate(col_keys):
        meta = classes[tabula_key]
        year = meta["year"]
        era = meta["era"]
        
        for row_idx, r_level in enumerate(retrofit_levels):
            # Calculate grid coordinates
            cx = origin[0] + col_idx * spacing_x
            cy = origin[1] + row_idx * spacing_y
            geom = box(cx - width / 2.0, cy - depth / 2.0, cx + width / 2.0, cy + depth / 2.0)
            
            identifier = f"urn:adv:oid:DEHHALKA_MFH_{tabula_key}_R{r_level}"
            building_name = f"MFH_{tabula_key}_Baujahr_{year}_Retrofit_{r_level}"
            
            # Populate exact schema matching buildings_unified.parquet
            row = {
                "identifier": identifier,
                "name": building_name,
                "beginnt": base_params["beginnt"],
                "advStandardModell": base_params["advStandardModell"],
                "gebaeudefunktion": base_params["gebaeudefunktion"],
                "building": base_params["building"],
                "building_type": base_params["building_type"],
                "art": base_params["art"],
                "anzahlDerOberirdischenGeschosse": base_params["anzahlDerOberirdischenGeschosse"],
                "anzahlDerUnterirdischenGeschosse": base_params["anzahlDerUnterirdischenGeschosse"],
                "dachform": base_params["dachform"],
                "dachart": base_params["dachart"],
                "bauweise": base_params["bauweise"],
                "hochhaus": base_params["hochhaus"],
                "weitereGebaeudefunktion": np.nan,
                "grundflaeche": ground_area,
                "grundflaeche_uom": base_params["grundflaeche_uom"],
                
                # Parametric Variations
                "baujahr": float(year),
                "retrofit": int(r_level),
                "tabula_class": tabula_key,
                "tabula_era": era,
                
                # Solar & Grid fields
                "eignung": base_params["eignung"],
                "solar_power_value": base_params["solar_power_value"],
                "solar_power_zone_total": base_params["solar_power_zone_total"],
                "solar_n_buildings_in_zone": base_params["solar_n_buildings_in_zone"],
                "solar_match_score": base_params["solar_match_score"],
                "solar_pvarea_value": base_params["solar_pvarea_value"],
                "solar_pvarea_zone_total": base_params["solar_pvarea_zone_total"],
                "elec_value": base_params["elec_value"],
                "elec_zone_total": base_params["elec_zone_total"],
                "elec_n_buildings_in_zone": base_params["elec_n_buildings_in_zone"],
                "elec_match_score": base_params["elec_match_score"],
                "heat_value": base_params["heat_value"],
                "heat_zone_total": base_params["heat_zone_total"],
                "heat_n_buildings_in_zone": base_params["heat_n_buildings_in_zone"],
                "heat_match_score": base_params["heat_match_score"],
                "solar_high_confidence": base_params["solar_high_confidence"],
                "elec_high_confidence": base_params["elec_high_confidence"],
                "heat_high_confidence": base_params["heat_high_confidence"],
                "geometry": geom,
            }
            records.append(row)
            
    return gpd.GeoDataFrame(records, geometry="geometry", crs=crs)

def validate_dataset(gdf: gpd.GeoDataFrame):
    """Verifies that all 30 buildings adhere to gis_preprocessor requirements."""
    assert len(gdf) == 30, f"Expected 30 buildings, found {len(gdf)}."
    
    # 1. Ensure footprint areas are identical and positive
    areas = gdf["grundflaeche"].unique()
    assert len(areas) == 1 and areas[0] > 0, f"Inconsistent or non-positive footprints: {areas}"
    
    # 2. Check complete TABULA classes coverage
    tabula_found = set(gdf["tabula_class"].unique())
    expected_classes = {"A", "B", "C", "D", "E", "F", "G", "H", "I", "J"}
    assert tabula_found == expected_classes, f"Missing classes: {expected_classes - tabula_found}"
    
    # 3. Check 3 retrofit levels per class
    for b_class, group in gdf.groupby("tabula_class"):
        levels = set(group["retrofit"].unique())
        assert levels == {0, 1, 2}, f"Class {b_class} does not contain retrofit levels [0, 1, 2]: {levels}"
        
    print(f"✓ Validation Successful: 30 MFH buildings correctly generated across TABULA classes A-J.")

def export_parquet_and_gml(gdf: gpd.GeoDataFrame, output_dir: str = "./data_output"):
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    
    # Export Parquet
    parquet_path = out / "buildings_unified_30_mfh.parquet"
    gdf.to_parquet(parquet_path, engine="pyarrow", compression="snappy")
    print(f"GeoParquet written to: {parquet_path.resolve()}")
    
    # Export GML
    gml_path = out / "buildings_unified_30_mfh.gml"
    gdf.to_file(gml_path, driver="GML")
    print(f"GML file written to:   {gml_path.resolve()}")

In [3]:
# Execute pipeline
mfh_30_gdf = generate_30_mfh_dataset(
    BASE_MFH_CONFIG,
    TABULA_MFH_CLASSES,
    RETROFIT_LEVELS,
    GRID_ORIGIN,
    GRID_SPACING_X,
    GRID_SPACING_Y,
    CRS
)

validate_dataset(mfh_30_gdf)
export_parquet_and_gml(mfh_30_gdf)

# Display summary table of the generated 30 buildings
mfh_30_gdf[["identifier", "tabula_class", "tabula_era", "baujahr", "retrofit", "grundflaeche"]].head(12)

✓ Validation Successful: 30 MFH buildings correctly generated across TABULA classes A-J.
GeoParquet written to: J:\Desktop\buildsim\districtgenerator_addons\data\data_output\buildings_unified_30_mfh.parquet
GML file written to:   J:\Desktop\buildsim\districtgenerator_addons\data\data_output\buildings_unified_30_mfh.gml


,identifier,tabula_class,tabula_era,baujahr,retrofit,grundflaeche
0,urn:adv:oid:DEHHALKA_MFH_A_R0,A,<= 1859,1850.0,0,300.0
1,urn:adv:oid:DEHHALKA_MFH_A_R1,A,<= 1859,1850.0,1,300.0
2,urn:adv:oid:DEHHALKA_MFH_A_R2,A,<= 1859,1850.0,2,300.0
3,urn:adv:oid:DEHHALKA_MFH_B_R0,B,1860 - 1918,1900.0,0,300.0
4,urn:adv:oid:DEHHALKA_MFH_B_R1,B,1860 - 1918,1900.0,1,300.0
5,urn:adv:oid:DEHHALKA_MFH_B_R2,B,1860 - 1918,1900.0,2,300.0
6,urn:adv:oid:DEHHALKA_MFH_C_R0,C,1919 - 1948,1930.0,0,300.0
7,urn:adv:oid:DEHHALKA_MFH_C_R1,C,1919 - 1948,1930.0,1,300.0
8,urn:adv:oid:DEHHALKA_MFH_C_R2,C,1919 - 1948,1930.0,2,300.0
9,urn:adv:oid:DEHHALKA_MFH_D_R0,D,1949 - 1957,1954.0,0,300.0
